In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize']= (12,6)

In [ ]:
# Load optimized data from pickle file (data types preserved)
import pickle

with open('../../data/train_optimized.pkl', 'rb') as f:
    df = pickle.load(f)

print("="*80)
print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
df = df.set_index('TransactionID') 

df.head()

In [ ]:
# List categorical columns (object and category types)
categorical_cols = [col for col in df.columns if df[col].dtype in ['object', 'category']]

# High cardinality categorical columns (>50 unique values)
categorical_but_high_cardinality = [col for col in categorical_cols 
                                    if df[col].nunique() > 50]

# Low cardinality categorical columns (<=50 unique values) - SEPARATED
categorical_cols_clean = [col for col in categorical_cols 
                          if col not in categorical_but_high_cardinality]

# All numeric columns
numeric_cols = [col for col in df.columns 
                if df[col].dtype in ['int64', 'float64', 'Int32', 'Int8', 'Int16']]

# Numeric columns that behave like categorical (<20 unique values)
numeric_but_categorical = [col for col in numeric_cols 
                          if df[col].nunique() < 20 
                          and col not in ['isFraud', 'TransactionID']]

# Pure numeric columns (continuous) - SEPARATED
numeric_cols_clean = [col for col in numeric_cols 
                      if col not in numeric_but_categorical 
                      and col not in ['isFraud', 'TransactionID']]

print("="*80)
print("COLUMN CLASSIFICATION")
print("="*80)

print(f"\n📌 CATEGORICAL COLUMNS:")
print(f"   - Total Categorical: {len(categorical_cols)}")
print(f"   - Low Cardinality (<=50 unique): {len(categorical_cols_clean)}")
print(f"   - High Cardinality (>50 unique): {len(categorical_but_high_cardinality)}")

print(f"\n📌 NUMERIC COLUMNS:")
print(f"   - Total Numeric: {len(numeric_cols)}")
print(f"   - Continuous (>=20 unique): {len(numeric_cols_clean)}")
print(f"   - Categorical-like (<20 unique): {len(numeric_but_categorical)}")

print(f"\n📌 TOTAL: {len(df.columns)} columns")
print("="*80)

# Display the column lists
print(f"\nCategorical (Low Cardinality): {categorical_cols_clean}")
print(f"\nCategorical (High Cardinality): {categorical_but_high_cardinality}")
print(f"\nNumeric (Categorical-like): {numeric_but_categorical}")

## 1. Target Distribution (Fraud Rate Analysis)

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
fraud_counts = df['isFraud'].value_counts()
axes[0].bar(['Not Fraud (0)', 'Fraud (1)'], fraud_counts.values, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Fraud Distribution (Count)', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Percentage pie chart
axes[1].pie(fraud_counts.values, labels=['Not Fraud', 'Fraud'], autopct='%1.2f%%', 
            colors=['#2ecc71', '#e74c3c'], explode=[0, 0.1])
axes[1].set_title('Fraud Distribution (%)', fontsize=14)

plt.tight_layout()
plt.show()

print(f"Fraud Rate: {df['isFraud'].mean()*100:.2f}%")
print(f"Class Imbalance Ratio: 1:{int((1-df['isFraud'].mean())/df['isFraud'].mean())}")

## 2. Feature Groups Overview (IEEE Dataset Structure)

In [ ]:
# Group columns by their prefix/category
feature_groups = {
    'Transaction Core': ['TransactionDT', 'TransactionAmt', 'ProductCD', 'isFraud'],
    'Card Info': [col for col in df.columns if col.startswith('card')],
    'Address': [col for col in df.columns if col.startswith('addr')],
    'Email Domain': [col for col in df.columns if 'emaildomain' in col.lower()],
    'M Features (Match)': [col for col in df.columns if col.startswith('M')],
    'C Features (Count)': [col for col in df.columns if col.startswith('C') and col[1:].isdigit()],
    'D Features (Timedelta)': [col for col in df.columns if col.startswith('D') and col[1:].isdigit()],
    'V Features (Vesta)': [col for col in df.columns if col.startswith('V')],
    'ID Features': [col for col in df.columns if col.startswith('id_')],
    'Device Info': [col for col in df.columns if 'device' in col.lower()],
}

print("="*80)
print("IEEE FRAUD DETECTION - FEATURE GROUPS")
print("="*80)

total = 0
group_sizes = {}
for group, cols in feature_groups.items():
    existing_cols = [c for c in cols if c in df.columns]
    group_sizes[group] = len(existing_cols)
    total += len(existing_cols)
    print(f"{group:25s}: {len(existing_cols):3d} columns")

print("="*80)
print(f"{'TOTAL':25s}: {total:3d} columns")

# Visualize group sizes
plt.figure(figsize=(12, 6))
groups = list(group_sizes.keys())
sizes = list(group_sizes.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(groups)))

bars = plt.barh(groups, sizes, color=colors)
plt.xlabel('Number of Features')
plt.title('Feature Distribution by Group')

for bar, size in zip(bars, sizes):
    plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
             f'{size}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Missing Values Analysis

In [ ]:
# Missing values by feature group
missing_by_group = {}

for group, cols in feature_groups.items():
    existing_cols = [c for c in cols if c in df.columns]
    if existing_cols:
        missing_pct = df[existing_cols].isnull().mean().mean() * 100
        missing_by_group[group] = missing_pct

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Missing by group
groups = list(missing_by_group.keys())
missing_pcts = list(missing_by_group.values())
colors = ['#e74c3c' if p > 50 else '#f39c12' if p > 20 else '#2ecc71' for p in missing_pcts]

bars = axes[0].barh(groups, missing_pcts, color=colors)
axes[0].set_xlabel('Missing Rate (%)')
axes[0].set_title('Missing Values by Feature Group')
axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')

for bar, pct in zip(bars, missing_pcts):
    axes[0].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
                 f'{pct:.1f}%', va='center')

# Top 20 columns with most missing
missing_cols = df.isnull().mean().sort_values(ascending=False).head(20) * 100
axes[1].barh(missing_cols.index, missing_cols.values, color='#e74c3c')
axes[1].set_xlabel('Missing Rate (%)')
axes[1].set_title('Top 20 Columns with Most Missing Values')

plt.tight_layout()
plt.show()

# Summary
print(f"\nColumns with >50% missing: {(df.isnull().mean() > 0.5).sum()}")
print(f"Columns with no missing: {(df.isnull().mean() == 0).sum()}")
print(f"Total missing cells: {df.isnull().sum().sum():,} ({df.isnull().mean().mean()*100:.1f}%)")

## 4. Transaction Amount Analysis

In [ ]:
# Transaction Amount distribution by fraud status
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution (log scale)
for fraud_val, color, label in [(0, '#2ecc71', 'Not Fraud'), (1, '#e74c3c', 'Fraud')]:
    data = df[df['isFraud'] == fraud_val]['TransactionAmt']
    axes[0, 0].hist(np.log1p(data), bins=50, alpha=0.6, color=color, label=label)
axes[0, 0].set_xlabel('Log(TransactionAmt + 1)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Transaction Amount Distribution (Log Scale)')
axes[0, 0].legend()

# Box plot by fraud
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[0, 1])
axes[0, 1].set_title('Transaction Amount by Fraud Status')
axes[0, 1].set_xlabel('isFraud')
axes[0, 1].set_ylabel('TransactionAmt')
plt.suptitle('')

# Summary statistics
fraud_amt = df[df['isFraud'] == 1]['TransactionAmt']
not_fraud_amt = df[df['isFraud'] == 0]['TransactionAmt']

stats_data = {
    'Metric': ['Mean', 'Median', 'Std', 'Min', 'Max', '25%', '75%'],
    'Not Fraud': [not_fraud_amt.mean(), not_fraud_amt.median(), not_fraud_amt.std(), 
                  not_fraud_amt.min(), not_fraud_amt.max(), not_fraud_amt.quantile(0.25), not_fraud_amt.quantile(0.75)],
    'Fraud': [fraud_amt.mean(), fraud_amt.median(), fraud_amt.std(),
              fraud_amt.min(), fraud_amt.max(), fraud_amt.quantile(0.25), fraud_amt.quantile(0.75)]
}

# Amount ranges fraud rate
bins = [0, 50, 100, 200, 500, 1000, 5000, 10000, float('inf')]
labels = ['0-50', '50-100', '100-200', '200-500', '500-1K', '1K-5K', '5K-10K', '10K+']
df['amt_bin'] = pd.cut(df['TransactionAmt'], bins=bins, labels=labels)

fraud_by_amt = df.groupby('amt_bin')['isFraud'].agg(['mean', 'count'])
fraud_by_amt['mean'] = fraud_by_amt['mean'] * 100

axes[1, 0].bar(fraud_by_amt.index.astype(str), fraud_by_amt['mean'], color='#3498db')
axes[1, 0].set_xlabel('Transaction Amount Range')
axes[1, 0].set_ylabel('Fraud Rate (%)')
axes[1, 0].set_title('Fraud Rate by Amount Range')
axes[1, 0].tick_params(axis='x', rotation=45)

# Transaction count by range
axes[1, 1].bar(fraud_by_amt.index.astype(str), fraud_by_amt['count'], color='#9b59b6')
axes[1, 1].set_xlabel('Transaction Amount Range')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Transaction Count by Amount Range')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Clean up temp column
df = df.drop('amt_bin', axis=1)

# Print stats
print("\nTransaction Amount Statistics:")
print(pd.DataFrame(stats_data).to_string(index=False))

## 5. Categorical Features - Fraud Rate Analysis

In [ ]:
# Key categorical features fraud analysis
key_cats = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'DeviceType']
existing_cats = [c for c in key_cats if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(existing_cats):
    # Calculate fraud rate per category
    fraud_rate = df.groupby(col)['isFraud'].agg(['mean', 'count']).reset_index()
    fraud_rate['mean'] = fraud_rate['mean'] * 100
    fraud_rate = fraud_rate.sort_values('mean', ascending=False).head(10)
    
    # Plot
    colors = plt.cm.RdYlGn_r(fraud_rate['mean'] / fraud_rate['mean'].max())
    bars = axes[idx].barh(fraud_rate[col].astype(str), fraud_rate['mean'], color=colors)
    axes[idx].set_xlabel('Fraud Rate (%)')
    axes[idx].set_title(f'{col} - Fraud Rate by Category')
    
    # Add count labels
    for bar, count in zip(bars, fraud_rate['count']):
        axes[idx].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                      f'n={count:,}', va='center', fontsize=8)

# Card features (card1-card6)
card_cols = [c for c in df.columns if c.startswith('card') and c in ['card1', 'card2', 'card3', 'card5']]
card_fraud_rates = {}

for col in card_cols:
    # Top 5 fraud rate categories for each card
    top_fraud = df.groupby(col)['isFraud'].mean().nlargest(5).mean() * 100
    card_fraud_rates[col] = top_fraud

axes[5].bar(card_fraud_rates.keys(), card_fraud_rates.values(), color='#e74c3c')
axes[5].set_ylabel('Avg Fraud Rate of Top 5 Categories (%)')
axes[5].set_title('Card Features - Fraud Potential')

plt.tight_layout()
plt.show()

## 6. Time Analysis (TransactionDT)

In [ ]:
# TransactionDT is seconds from a reference datetime
# Extract time features
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day'] = (df['TransactionDT'] // (3600 * 24)) % 7
df['week'] = df['TransactionDT'] // (3600 * 24 * 7)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Fraud rate by hour
hourly_fraud = df.groupby('hour')['isFraud'].mean() * 100
axes[0, 0].plot(hourly_fraud.index, hourly_fraud.values, marker='o', color='#e74c3c', linewidth=2)
axes[0, 0].fill_between(hourly_fraud.index, hourly_fraud.values, alpha=0.3, color='#e74c3c')
axes[0, 0].set_xlabel('Hour of Day')
axes[0, 0].set_ylabel('Fraud Rate (%)')
axes[0, 0].set_title('Fraud Rate by Hour of Day')
axes[0, 0].set_xticks(range(0, 24, 2))
axes[0, 0].grid(True, alpha=0.3)

# Transaction volume by hour
hourly_count = df.groupby('hour').size()
axes[0, 1].bar(hourly_count.index, hourly_count.values, color='#3498db', alpha=0.7)
axes[0, 1].set_xlabel('Hour of Day')
axes[0, 1].set_ylabel('Transaction Count')
axes[0, 1].set_title('Transaction Volume by Hour')

# Fraud rate by day of week
daily_fraud = df.groupby('day')['isFraud'].mean() * 100
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 0].bar(range(7), daily_fraud.values, color='#9b59b6')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Fraud Rate (%)')
axes[1, 0].set_title('Fraud Rate by Day of Week')
axes[1, 0].set_xticks(range(7))
axes[1, 0].set_xticklabels(days)

# Fraud over time (weeks)
weekly_fraud = df.groupby('week')['isFraud'].mean() * 100
axes[1, 1].plot(weekly_fraud.index, weekly_fraud.values, color='#e74c3c', linewidth=2)
axes[1, 1].set_xlabel('Week Number')
axes[1, 1].set_ylabel('Fraud Rate (%)')
axes[1, 1].set_title('Fraud Rate Over Time (Weekly)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Clean up temp columns
df = df.drop(['hour', 'day', 'week'], axis=1)

print(f"\nPeak fraud hour: {hourly_fraud.idxmax()}:00 ({hourly_fraud.max():.2f}%)")
print(f"Lowest fraud hour: {hourly_fraud.idxmin()}:00 ({hourly_fraud.min():.2f}%)")

## 7. V Features Correlation Analysis (Vesta Engineered)

In [ ]:
# V features are Vesta-engineered features (V1-V339)
v_cols = [col for col in df.columns if col.startswith('V')]
print(f"Total V features: {len(v_cols)}")

# Correlation with target
v_target_corr = df[v_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 V features correlated with fraud
top_v = v_target_corr.head(20)
axes[0].barh(top_v.index, top_v.values, color='#e74c3c')
axes[0].set_xlabel('Absolute Correlation with isFraud')
axes[0].set_title('Top 20 V Features Correlated with Fraud')
axes[0].invert_yaxis()

# V features correlation distribution
axes[1].hist(v_target_corr.values, bins=50, color='#3498db', edgecolor='white')
axes[1].set_xlabel('Absolute Correlation')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of V Feature Correlations with Fraud')
axes[1].axvline(x=0.1, color='red', linestyle='--', label='0.1 threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Group V features by correlation strength
high_corr_v = v_target_corr[v_target_corr > 0.1].index.tolist()
med_corr_v = v_target_corr[(v_target_corr > 0.05) & (v_target_corr <= 0.1)].index.tolist()
low_corr_v = v_target_corr[v_target_corr <= 0.05].index.tolist()

print(f"\nV Features by Correlation Strength:")
print(f"  High (>0.1): {len(high_corr_v)} features")
print(f"  Medium (0.05-0.1): {len(med_corr_v)} features")
print(f"  Low (<0.05): {len(low_corr_v)} features")
print(f"\nTop 5 V features: {list(top_v.head(5).index)}")

## 8. Identity Features Analysis

In [ ]:
# Identity features (id_01 - id_38)
id_cols = [col for col in df.columns if col.startswith('id_')]
print(f"Total ID features: {len(id_cols)}")

# Check which ones are numeric vs categorical
id_numeric = [col for col in id_cols if df[col].dtype in ['int64', 'float64', 'Int8', 'Int16', 'Int32']]
id_categorical = [col for col in id_cols if col not in id_numeric]

print(f"Numeric ID features: {len(id_numeric)}")
print(f"Categorical ID features: {len(id_categorical)}")

# Correlation of numeric ID features with fraud
if id_numeric:
    id_corr = df[id_numeric + ['isFraud']].corr()['isFraud'].drop('isFraud').abs().sort_values(ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Correlation
    axes[0].barh(id_corr.index, id_corr.values, color='#9b59b6')
    axes[0].set_xlabel('Absolute Correlation with isFraud')
    axes[0].set_title('ID Features Correlation with Fraud')
    axes[0].invert_yaxis()
    
    # Missing rate
    id_missing = df[id_cols].isnull().mean().sort_values(ascending=False) * 100
    axes[1].barh(id_missing.index, id_missing.values, color='#e74c3c')
    axes[1].set_xlabel('Missing Rate (%)')
    axes[1].set_title('ID Features Missing Rate')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()

# Device analysis
if 'DeviceType' in df.columns:
    print("\nDevice Type Analysis:")
    device_fraud = df.groupby('DeviceType')['isFraud'].agg(['mean', 'count'])
    device_fraud['mean'] = device_fraud['mean'] * 100
    print(device_fraud.to_string())

## 9. Feature Importance Summary & Recommendations

In [ ]:
# Overall correlation analysis - Top features
numeric_df = df.select_dtypes(include=[np.number])
all_corr = numeric_df.corr()['isFraud'].drop('isFraud').abs().sort_values(ascending=False)

# Top 30 features
top_30 = all_corr.head(30)

plt.figure(figsize=(12, 10))
colors = ['#e74c3c' if 'V' in idx else '#3498db' if 'C' in idx else '#2ecc71' if 'D' in idx else '#9b59b6' for idx in top_30.index]
plt.barh(top_30.index, top_30.values, color=colors)
plt.xlabel('Absolute Correlation with isFraud')
plt.title('Top 30 Features Correlated with Fraud')
plt.gca().invert_yaxis()

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='V Features'),
                   Patch(facecolor='#3498db', label='C Features'),
                   Patch(facecolor='#2ecc71', label='D Features'),
                   Patch(facecolor='#9b59b6', label='Other')]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

# Summary
print("="*80)
print("EDA SUMMARY & RECOMMENDATIONS")
print("="*80)
print(f"""
📊 DATASET OVERVIEW:
   - Total Features: {len(df.columns)}
   - Fraud Rate: {df['isFraud'].mean()*100:.2f}% (Highly Imbalanced)
   
🔑 KEY FINDINGS:
   1. V features (Vesta) are most predictive - {len([c for c in top_30.index if 'V' in c])} in top 30
   2. C features (Count) show strong fraud signals
   3. Time features reveal fraud patterns by hour
   4. High missing rate in identity features (~40%)
   
⚡ RECOMMENDATIONS FOR MODELING:
   1. Use class_weight='balanced' or SMOTE for imbalance
   2. Focus on V features for feature selection
   3. Create time-based features (hour, day_of_week)
   4. Handle missing values carefully (especially identity)
   5. Use TimeSeriesSplit for validation (time-based data)
   
🎯 POTENTIAL NEW FEATURES:
   - Transaction amount bins
   - Hour of transaction
   - Email domain matching (P vs R)
   - Card info aggregations
""")
print("="*80)